# Gaussian mixture models
The mixture density, two-step sampling, responsibilities for the seven-point example of *Mathematics for
Machine Learning* (Section 11.2), the slope-zero mean, the collapse of maximum likelihood, and GMM versus k-means.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy import stats
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

np.set_printoptions(precision=4, suppress=True)
# one seeded random generator for the whole notebook
rng = np.random.default_rng(1)

## 1. The mixture density 0.5 N(-2, 0.5) + 0.2 N(1, 2) + 0.3 N(4, 1)

In [ ]:
pi = np.array([0.5, 0.2, 0.3])
mu = np.array([-2.0, 1.0, 4.0])
var = np.array([0.5, 2.0, 1.0])
# weighted components at x = 0 (scale = standard deviation)
weighted = pi * stats.norm(mu, np.sqrt(var)).pdf(0)
print("weighted components at 0:", weighted, " p(0) =", weighted.sum().round(4))
# the total area is 1
g = np.linspace(-10, 12, 20001)
dens = (pi * stats.norm(mu, np.sqrt(var)).pdf(g[:, None])).sum(axis=1)
print("area under p(x):", np.trapezoid(dens, g).round(6))

## 2. Sampling in two steps

In [ ]:
# step 1: pick a component for every draw
z = rng.choice(3, size=3000, p=pi)
# step 2: draw from the picked component
x = rng.normal(mu[z], np.sqrt(var[z]))
print("share of draws from each component:", np.bincount(z) / len(z))
fig = go.Figure(go.Histogram(x=x, histnorm="probability density", nbinsx=60, name="draws"))
fig.add_trace(go.Scatter(x=g, y=dens, mode="lines", name="mixture density"))
fig.update_layout(template="simple_white", xaxis_range=[-6, 8], height=380)
fig.show()

## 3. Responsibilities for the seven-point example

In [ ]:
X7 = np.array([-3, -2.5, -1, 0, 2, 4, 5.0])
pi7, mu7, var7 = np.ones(3) / 3, np.array([-4.0, 0.0, 8.0]), np.array([1.0, 0.2, 3.0])
# weighted density of every point under every component, shape (7, 3)
W = pi7 * stats.norm(mu7, np.sqrt(var7)).pdf(X7[:, None])
# responsibilities: each row divided by its sum
R = W / W.sum(axis=1, keepdims=True)
print(R)
print("row sums:", R.sum(axis=1))
print("N_k:", R.sum(axis=0).round(3), " total:", R.sum())
print("log-likelihood of the starting mixture:", np.log(W.sum(axis=1)).sum().round(2))

## 4. The slope-zero mean of component 1

In [ ]:
mu1_new = (R[:, 0] * X7).sum() / R[:, 0].sum()
print("mu_1 = sum(r x) / N_1 =", mu1_new.round(3))
# after moving mu_1 the responsibilities change, so mu_1 no longer satisfies its own equation
mu_moved = mu7.copy()
mu_moved[0] = mu1_new
W2 = pi7 * stats.norm(mu_moved, np.sqrt(var7)).pdf(X7[:, None])
R2 = W2 / W2.sum(axis=1, keepdims=True)
print("new responsibilities of component 1:", R2[:, 0])
print("weighted mean with the new responsibilities:", ((R2[:, 0] * X7).sum() / R2[:, 0].sum()).round(3))

## 5. Maximum likelihood breaks: a component collapsing onto one point
Component 1 sits on the point -3; components 2 and 3 are N(0, 1) and N(4, 1); equal weights.

In [ ]:
for s1 in (1, 0.1, 0.01, 0.001):
    sd = np.array([s1, 1, 1])
    ll = np.log((np.ones(3) / 3 * stats.norm([-3, 0, 4], sd).pdf(X7[:, None])).sum(axis=1)).sum()
    print(f"sigma_1 = {s1}: log-likelihood {ll:.2f}")

## 6. Real data: Old Faithful waiting times, and Iris
Old Faithful: one normal versus a mixture of two. The file was saved from seaborn's `geyser` dataset.

In [ ]:
import pandas as pd
wait = pd.read_csv("data/old_faithful.csv")["waiting"].to_numpy(float)
ll_one = stats.norm(wait.mean(), wait.std()).logpdf(wait).sum()
g2 = GaussianMixture(2, random_state=0).fit(wait[:, None])
print("one normal: mean", wait.mean().round(1), "std", wait.std().round(1), " log-likelihood", round(ll_one, 1))
print("mixture: weights", g2.weights_.round(2), "means", g2.means_.ravel().round(1),
      "std", np.sqrt(g2.covariances_.ravel()).round(1), " log-likelihood", round(g2.score(wait[:, None]) * len(wait), 1))

Iris: k-means against a GMM, then the GMM with only the covariance shape changed.

In [ ]:
from sklearn.datasets import load_iris
X, species = load_iris(return_X_y=True)
print("k-means ARI:", round(adjusted_rand_score(species, KMeans(3, n_init=10, random_state=0).fit_predict(X)), 2))
for ct in ("full", "spherical"):
    gm = GaussianMixture(3, covariance_type=ct, n_init=10, random_state=0).fit(X)
    print(f"GMM {ct} ARI:", round(adjusted_rand_score(species, gm.predict(X)), 2))